## Fine-Tuning GPT-2 on Poetry

Tutorial: [GeeksForGeeks](https://www.geeksforgeeks.org/deep-learning/how-to-fine-tune-an-llm-from-hugging-face/)

In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from datasets import load_dataset

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')

dataset = load_dataset("biglam/gutenberg-poetry-corpus")

e:\GitHub\University_projects\semester 4\AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6793.96it/s]


In [2]:
def tokenize_function(examples):
	outputs = tokenizer(
		examples["line"],        # column name in the dataset
		truncation=True,         # cuts if too long
		padding="max_length",    # pads if too short
		max_length=128
	)
	outputs["labels"] = outputs["input_ids"].copy()
		# mask padding tokens in labels with -100
		# -100 tells PyTorch to IGNORE these in loss calculation
	outputs["labels"] = [
		[-100 if token == tokenizer.pad_token_id else token for token in label]
		for label in outputs["labels"]
	]
	return outputs

If we keep the pading tokens in the labels, the model will learn to predict them. Meaning it will waste time for predicting nothing!

In [3]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [4]:
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [6]:
from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./gpt2-poetry",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,  # optional boost
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [7]:
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
	print("Using CPU")
print(torch.version.cuda)

Using GPU: NVIDIA GeForce GTX 1650
12.1


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(1000)),  # smaller subset for demo
    data_collator=data_collator
)

In [12]:
trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


TrainOutput(global_step=96, training_loss=3.785440444946289, metrics={'train_runtime': 1062.5563, 'train_samples_per_second': 2.823, 'train_steps_per_second': 0.09, 'total_flos': 195969024000000.0, 'train_loss': 3.785440444946289, 'epoch': 3.0})

In [20]:
def complete_poem(poem, temperature=0.7, repetition_penalty=1.3, max_new_tokens=20):
	context = ""

	for line in poem:
		print(line)
		if line != "<<BLANK>>":
			context += line + "\n"
		else:
			tokenized_context = tokenizer(context, return_tensors="pt").to(model.device)
			# the new token with the predicted text
			generated_token = model.generate(
				**tokenized_context,
				max_new_tokens=max_new_tokens,
				repetition_penalty=repetition_penalty,
				temperature=temperature,
				do_sample=True,
				pad_token_id=tokenizer.eos_token_id
			)
			# we don't want to have the hole text, just the new part
			input_size = tokenized_context['input_ids'].shape[1]
			new_tokens = generated_token[0][input_size:] # just what GPT generated
			decoded_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

			context += decoded_text + '\n'
	return context

In [21]:
test_poem = [
  	"From fairest creatures we desire increase,",
	"<<BLANK>>", #"That thereby beauty's rose might never die,"
	"But as the riper should by time decease,",
	"<<BLANK>>", #"His tender heir might bear his memory:"
	"But thou contracted to thine own bright eyes,",
	"Feed'st thy light's flame with self-substantial fuel,",
	"<<BLANK>>", #"Making a famine where abundance lies,"
	"<<BLANK>>", #"Thy self thy foe, to thy sweet self too cruel:"
	"Thou that art now the world's fresh ornament,",
	"And only herald to the gaudy spring,",
	"<<BLANK>>", #"Within thine own bud buriest thy content,"
	"And, tender churl, mak'st waste in niggarding:",
	"<<BLANK>>", #"Pity the world, or else this glutton be,"
	"To eat the world's due, by the grave and thee.",
]

In [22]:
predicted_poems = [complete_poem(test_poem, temperature=i) for i in [0.3, 0.6, 0.9, 1.2, 1.5]]

From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
<<BLANK>>
<<BLANK>>
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
<<BLANK>>
And, tender churl, mak'st waste in niggarding:
<<BLANK>>
To eat the world's due, by the grave and thee.
From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
<<BLANK>>
<<BLANK>>
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
<<BLANK>>
And, tender churl, mak'st waste in niggarding:
<<BLANK>>
To eat the world's due, by the grave and thee.
From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st 

In [23]:
for poem in predicted_poems:
	print("-" * 10)
	print(poem)

----------
From fairest creatures we desire increase,
And in the wildest forest; and I will not be to thee. O my son! thou
But as the riper should by time decease,
The dusky river-crested me with its stream--I am a stranger here!"


But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
'tis he who is born of water? And how shall they live together?" said she: "
"O your mother," cried out again."My daughter was dead;" but her father still lived on
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
For all his songs are sung at home.' The child answered:" 'Til you see it,'
And, tender churl, mak'st waste in niggarding:
'Twice from each tree there lies an arrowdance!'—a song for him whom no
To eat the world's due, by the grave and thee.

----------
From fairest creatures we desire increase,
. .--And the birds in their nests! I will find them; and they shall hear us
But as the riper should by time decease,
Oh-woo

In [24]:
trainer.save_model("./gpt2-poetry")
tokenizer.save_pretrained("./gpt2-poetry")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]


('./gpt2-poetry\\tokenizer_config.json', './gpt2-poetry\\tokenizer.json')

### Conclusion

The model is a lot better! It can predict the next line of the poem relativly well. The best part is that the model it not a priest anymore, or at least not as much as it was before. To the naked eye it's a lot better.

In [31]:
test_poem_ro = [
  	"De-aș avea și eu o floare",
	"Mândră, dulce, răpitoare,",
	"<<BLANK>>", # "Ca și florile din mai,"
	"Fiice dulce-a unui plai,",
	"<<BLANK>>", # "Plai râzând cu iarbă verde,"
	"Ce se leagănă, se pierde,",
	"<<BLANK>>", # "Undoind încetișor,"
	"Șoptind șoapte de amor;"
]

In [32]:
predicted_poem_ro = complete_poem(test_poem_ro, temperature=0.7, repetition_penalty=1.3, max_new_tokens=20)

De-aș avea și eu o floare
Mândră, dulce, răpitoare,
<<BLANK>>
Fiice dulce-a unui plai,
<<BLANK>>
Ce se leagănă, se pierde,
<<BLANK>>
Șoptind șoapte de amor;


In [33]:
print(predicted_poem_ro)

De-aș avea și eu o floare
Mândră, dulce, răpitoare,
I am in the forest! I have seen you; see me now. O my mother--who
Fiice dulce-a unui plai,
Alyonahwisme mamum and roatia—no wonder there is no
Ce se leagănă, se pierde,
O king's son!" he exclaimed,— "Gentlemen of all nations are afraid to tell your
Șoptind șoapte de amor;

